# One-time official DINOv3 cache

Authenticates to the gated official model only if needed and downloads the pretrained snapshot once. It does not train anything.

Run the single code cell below with the **Python (skin)** kernel.

In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

from pathlib import Path

from notebook_config import HF_MODEL_SOURCE

MODEL_FILES = ["*.json", "*.safetensors"]

try:
    from huggingface_hub import login, snapshot_download
except ImportError as error:
    raise RuntimeError(
        f"huggingface_hub is missing. Run: {sys.executable} -m pip install huggingface_hub"
    ) from error

source = Path(HF_MODEL_SOURCE)
if source.is_dir():
    snapshot = source.resolve()
    print(f"Using existing local model directory: {snapshot}")
else:
    try:
        snapshot = Path(snapshot_download(
            repo_id=HF_MODEL_SOURCE,
            allow_patterns=MODEL_FILES,
            local_files_only=True,
        ))
        print(f"DINOv3 is already cached locally: {snapshot}")
    except Exception:
        print("The official DINOv3 model is gated. First request/accept access in your browser:")
        print(f"https://huggingface.co/{HF_MODEL_SOURCE}")
        print("Then enter a Hugging Face READ token below. The token input is hidden.")
        login(add_to_git_credential=False)
        snapshot = Path(snapshot_download(
            repo_id=HF_MODEL_SOURCE,
            allow_patterns=MODEL_FILES,
        ))

from transformers import AutoConfig
config = AutoConfig.from_pretrained(str(snapshot), local_files_only=True)
if int(config.hidden_size) != 768:
    raise ValueError(f"Expected DINOv3 ViT-B hidden size 768; observed {config.hidden_size}")
print(f"\nPASS: cached {HF_MODEL_SOURCE}")
print(f"Local snapshot: {snapshot}")
print("No dataset image was read and no model was trained. Notebook 03 can now run offline.")
